<a href="https://colab.research.google.com/github/Nurfinkaa/MachineLearning/blob/main/js6/TG6_3_ANN_244107020211_NURFINKA_LAILASARI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Praktikum 3**

**Install hnswlib terlebih dahulu**

In [ ]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=e1ceba33ed86794fe416b73c22b6b4b8e26925affb700567a0785c2c1f085e5d
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


**Percobaan berikut akan membandingkan exact NN dengan HNSW pada 1000 data 2D.**

In [ ]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[252 440 132  49 521]]
Distances: [[0.0099303  0.03158398 0.03967391 0.04006198 0.04027162]]
Waktu: 0.04988384246826172 detik

=== HNSW ===
Indices: [[252 440 132  49 521]]
Distances: [[9.8610828e-05 9.9754799e-04 1.5740190e-03 1.6049619e-03 1.6218033e-03]]
Waktu: 0.00024199485778808594 detik


**Lakukan percobaan pada metric distance yang berbeda, 1000 vs 1jt data, 2D vs 5D data. catat hasilnya pada tabel yang anda buat sendiri seperti pada praktikum 1.**

In [ ]:
import hnswlib
import numpy as np
import time
import pandas as pd

def exact_knn(X, q, space, k):
    if space == 'l2':
        s = ((X - q) ** 2).sum(axis=1)
    elif space == 'cosine':
        s = 1 - (X @ q) / (np.linalg.norm(X, axis=1) * np.linalg.norm(q))
    else:  # 'ip' (inner product): makin besar makin dekat
        s = 1 - (X @ q)
    return np.argsort(s)[:k]

def run_experiment(space, n_points, dim, k=5, n_query=50, seed=42, M=16, ef_construction=100, ef=50):
    rng = np.random.default_rng(seed)
    data = rng.random((n_points, dim)).astype(np.float32)
    query = np.full((1, dim), 0.5, dtype=np.float32)          # query seperti di modul
    queries = rng.random((n_query, dim)).astype(np.float32)   # untuk menghitung recall

    # --- Build index HNSW (dicatat terpisah) ---
    t0 = time.perf_counter()
    p = hnswlib.Index(space=space, dim=dim)
    p.init_index(max_elements=n_points, ef_construction=ef_construction, M=M, random_seed=seed)
    p.add_items(data)
    p.set_ef(ef)
    t_build = time.perf_counter() - t0

    # --- Waktu 1 query: Exact NN vs HNSW ---
    t0 = time.perf_counter()
    e0 = exact_knn(data, query[0], space, k)
    t_exact = time.perf_counter() - t0
    t0 = time.perf_counter()
    labels, _ = p.knn_query(query, k=k)
    t_hnsw = time.perf_counter() - t0

    # --- Recall rata-rata dari n_query query ---
    recalls = []
    for q in queries:
        e = exact_knn(data, q, space, k)
        a = p.knn_query(q.reshape(1, -1), k=k)[0][0]
        recalls.append(len(set(e) & set(a)) / k)

    nama = {"l2": "L2 (Euclidean)", "cosine": "Cosine", "ip": "Inner Product"}[space]
    return {"Distance Metrics": nama, "Dimensi": f"{dim}D", "Jumlah data": n_points,
            "Index ENN": [int(i) for i in e0], "Index HNSW": [int(i) for i in labels[0]],
            "Waktu ENN (ms)": round(t_exact * 1000, 4), "Waktu HNSW (ms)": round(t_hnsw * 1000, 4),
            "Recall rata-rata": round(float(np.mean(recalls)), 2), "Waktu build (s)": round(t_build, 3)}


In [ ]:
# ---- Jalankan semua kombinasi: 3 metric x 2 ukuran data x 2 dimensi ----
hasil = []
for space in ["l2", "cosine", "ip"]:
    for n in [1000, 1_000_000]:
        for d in [2, 5]:
            print("Menjalankan:", space, n, f"{d}D", flush=True)
            hasil.append(run_experiment(space, n, d))

df = pd.DataFrame(hasil)

Menjalankan: l2 1000 2D
Menjalankan: l2 1000 5D
Menjalankan: l2 1000000 2D
Menjalankan: l2 1000000 5D
Menjalankan: cosine 1000 2D
Menjalankan: cosine 1000 5D
Menjalankan: cosine 1000000 2D
Menjalankan: cosine 1000000 5D
Menjalankan: ip 1000 2D
Menjalankan: ip 1000 5D
Menjalankan: ip 1000000 2D
Menjalankan: ip 1000000 5D


In [ ]:
# ---- Cetak tabel format praktikum 1 ----
def fmt_index(lst):
    return "[" + " ".join(str(i) for i in lst) + "]"

print("\n" + "-" * 150)
print(f"{'Distance Metrics':<17}{'Dimensi':<9}{'Jumlah data':<13}{'Hasil Index terdekat ENN vs HNSW':<80}{'Waktu (ms) ENN , HNSW':<25}{'Recall'}")
print("-" * 150)
for _, r in df.iterrows():
    idx = f"{fmt_index(r['Index ENN'])}, {fmt_index(r['Index HNSW'])}"
    waktu = f"{r['Waktu ENN (ms)']} , {r['Waktu HNSW (ms)']}"
    print(f"{r['Distance Metrics']:<17}{r['Dimensi']:<9}{r['Jumlah data']:<13}{idx:<80}{waktu:<25}{r['Recall rata-rata']}")


------------------------------------------------------------------------------------------------------------------------------------------------------
Distance Metrics Dimensi  Jumlah data  Hasil Index terdekat ENN vs HNSW                                                Waktu (ms) ENN , HNSW    Recall
------------------------------------------------------------------------------------------------------------------------------------------------------
L2 (Euclidean)   2D       1000         [843 634 372 70 737], [843 634 372 70 737]                                      1.3226 , 0.2657          1.0
L2 (Euclidean)   5D       1000         [694 534 99 859 766], [694 534 99 859 766]                                      0.2624 , 0.1609          1.0
L2 (Euclidean)   2D       1000000      [19638 786703 328755 620323 897494], [19638 786703 328755 620323 897494]        134.0046 , 0.1706        1.0
L2 (Euclidean)   5D       1000000      [289393 11237 247737 16575 776364], [289393 11237 247737 16575 

In [ ]:
print("\nWaktu build index (detik):")
display(df[["Distance Metrics", "Dimensi", "Jumlah data", "Waktu build (s)"]])


Waktu build index (detik):


,Distance Metrics,Dimensi,Jumlah data,Waktu build (s)
0,L2 (Euclidean),2D,1000,0.153
1,L2 (Euclidean),5D,1000,0.076
2,L2 (Euclidean),2D,1000000,71.657
3,L2 (Euclidean),5D,1000000,109.451
4,Cosine,2D,1000,0.068
5,Cosine,5D,1000,0.060
6,Cosine,2D,1000000,110.734
7,Cosine,5D,1000000,107.760
8,Inner Product,2D,1000,0.048
9,Inner Product,5D,1000,0.041
